# Seminar 7 — Part 2: Monte Carlo simulation and diffuse scattering

In this notebook you will:
1. Set up a larger supercell with the **J values fitted in Notebook 1**.
2. Run a Monte Carlo simulation and save a trajectory.
3. Calculate the diffuse scattering from the simulated configurations using a gemmi-based code.
4. Average over many snapshots to obtain a clean diffuse scattering pattern.

The diffuse scattering is computed in the **hk0 plane** (l = 0 section of reciprocal space).

## Setup

In [ ]:
import importlib, subprocess, sys, os

# Notebook 2 does not use orb-models — no version check needed here.
# gemmi, ase, h5py must be present (h5py is required by CalculateScattering).

def _pkg_missing(name):
    return importlib.util.find_spec(name) is None

if any(_pkg_missing(p) for p in ['gemmi', 'ase', 'h5py']):
    print('Installing packages (Colab / first run) ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'ase', 'gemmi', 'h5py'], check=True)
    if not os.path.exists('seminar_disorder_2026'):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/aglie/seminar_disorder_2026.git'], check=True)
    os.chdir('seminar_disorder_2026')
    print('Done.')
else:
    print('All libraries found locally.')

import numpy as np
import matplotlib.pyplot as plt

from mc_helpers import (
    calculate_connectivity_lists,
    initialise,
    calc_E,
    monte_carlo_run,
    convert_spins_to_crystal_structure,
)
from CalculateScattering import Grid

print('Imports OK.')

## Section 1: Lattice and simulation parameters

Use the **same `cell` and `sites`** as in Notebook 1.  
For the J values, either:
- load them from `fitted_J.npy` (if you ran Notebook 1), or
- enter them manually in the `J_list` line.

The supercell here is **larger** than in Notebook 1 so the diffuse scattering has enough resolution.  
A box of 20×20×5 for a BCC-like structure gives ~4000 atoms and a 40×40-pixel diffuse pattern (better with averaging).  
Increase `box` if you want a sharper pattern and have time to wait.

In [ ]:
# ── Edit these to match your structure (same as Notebook 1) ─────────────────
cell = np.array([4.0, 4.0, 4.0, 90., 90., 90.])

sites = np.array([[0.0, 0.0, 0.0],
                  [0.0, 0.5, 0.5,],
                  [0.5, 0.0, 0.5,],
                  [0.5, 0.5, 0.0,]])

up_element   = 'Au'
down_element = 'Cu'

# Load J from Notebook 1, or set manually:
try:
    J_list = np.load('fitted_J.npy')
    print(f'Loaded J from Notebook 1: {J_list}')
except FileNotFoundError:
    J_list = np.array([1.0, 0.0])   # ← replace with your fitted values
    print(f'fitted_J.npy not found — using J_list = {J_list}')

# Supercell for the MC simulation + diffuse scattering
box = np.array([10, 10, 5], dtype=int)

# Composition
P_up = 0.5   # fraction of spin-up atoms

# MC temperature
kB      = 8.617333e-5   # eV / K
T_K     = 2320.0        # ← set your temperature in Kelvin
T       = T_K * kB      # converted to eV for the MC code

# MC parameters
N_equil    = 1000    # equilibration steps
N_prod     = 10000   # production steps
N_moves    = 1000    # attempted swaps per step
save_every = 10      # save a snapshot every this many production steps
# ────────────────────────────────────────────────────────────────────────────

connectivity_lists, frac_coords, dists, counts, N_ats = calculate_connectivity_lists(
    cell, sites, box, J_list
)

print(f'\nSupercell: {box[0]}×{box[1]}×{box[2]},  {N_ats} atoms')
print(f'J_list = {J_list}')
print(f'T = {T_K} K  =  {T:.4f} eV')

## Section 2: Monte Carlo simulation

We run the same Metropolis swap-move MC as in Seminar 6.  
After equilibration, we save a snapshot of the spin configuration every `save_every` steps to use for diffuse scattering averaging.

In [ ]:
np.random.seed(42)

spins, N_up = initialise(N_ats, P_up, J_list, connectivity_lists)
E_cur = calc_E(spins, N_ats, P_up, J_list, connectivity_lists)

# ── Equilibration ────────────────────────────────────────────────────────────
print(f'Equilibrating  ({N_equil} steps × {N_moves} moves) ...')
for _ in range(N_equil):
    spins, dE = monte_carlo_run(spins, connectivity_lists, J_list, T, N_moves)
    E_cur += dE
print('Equilibration done.')

# ── Production run ───────────────────────────────────────────────────────────
print(f'Production run  ({N_prod} steps × {N_moves} moves) ...')
history = []      # saved spin snapshots
E_trace = []      # energy per atom at each step

for step in range(N_prod):
    spins, dE = monte_carlo_run(spins, connectivity_lists, J_list, T, N_moves)
    E_cur += dE
    E_trace.append(E_cur / N_ats)

    if step % save_every == 0:
        history.append(spins.copy())

    if step % 50 == 0:
        print(f'  step {step:4d}/{N_prod}   E/atom = {E_cur/N_ats:.5f}')

print(f'\nSaved {len(history)} snapshots for diffuse scattering.')

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(E_trace)
plt.xlabel('Production step')
plt.ylabel('E / atom  (eV)')
plt.title('Energy during production run')
plt.tight_layout()
plt.show()

In [ ]:
# Save the final configuration as a CIF to inspect in VESTA
from mc_helpers import write_cif
os.makedirs('output', exist_ok=True)
write_cif(frac_coords, spins, cell, box, filename='output/final_config.cif',
          up_element=up_element, down_element=down_element)
print('Saved output/final_config.cif  — open in VESTA to inspect the ordering.')

## Section 3: Diffuse scattering

Diffuse scattering arises from short-range correlations in the disorder: if unlike atoms tend to sit next to each other (or avoid each other), the time-averaged structure factor will show intensity at positions between Bragg peaks.

We calculate the intensity in the **hk0** plane of reciprocal space using `CalculateScattering.py`, which places electron density on a grid and Fourier-transforms it with gemmi.

### Grid setup

The grid covers $h \in [-ll, +ll]$ and $k \in [-ll, +ll]$ in reciprocal lattice units, with step size $1/\text{box}$ (one step per unit cell).  
The $l$ axis has only two layers: $l = -1$ (index 0) and $l = 0$ (index 1, i.e. **hk0**).  
We always extract `z_layer=1`.

> **Why l = −1 and l = 0 only?**  The FFT in gemmi requires at least 2 points along each axis; using exactly 2 layers is the minimum and keeps memory manageable.

In [ ]:
# ── Grid setup (do not change — matches 2025 convention exactly) ─────────────
ll = 2   # reciprocal-space range: covers h,k in [-ll, +ll]

grid = Grid(
    lower_limits=[-ll,          -ll,          -1],
    step_sizes  =[ 1/box[0],     1/box[1],      1],
    no_pixels   =[ ll*2*box[0],  ll*2*box[1],   2],
)

# hk0 is at z_layer = 1  (l = lower_limits[2] + 1*step_sizes[2] = -1 + 1 = 0)
Z_LAYER = 1

print(f'Grid covers h ∈ [{grid.lower_limits[0]}, {-grid.lower_limits[0]}]  '
      f'k ∈ [{grid.lower_limits[1]}, {-grid.lower_limits[1]}]')
print(f'Pixels: {grid.no_pixels[0]} × {grid.no_pixels[1]}')
print(f'Resolution: {grid.step_sizes[0]:.4f} r.l.u./pixel')

### Single snapshot

In [ ]:
def calc_diffuse_hk0(spins_snapshot, frac_coords, cell, box,
                     grid, z_layer=Z_LAYER,
                     blur=0.01, up_element='Au', down_element='Cu'):
    """Calculate diffuse scattering intensity in the hk0 plane for one snapshot."""
    crystal = convert_spins_to_crystal_structure(
        frac_coords, spins_snapshot, cell, box,
        up_element=up_element, down_element=down_element
    )
    sf = crystal.calculate_scattering(grid, blur=blur)
    return sf.get_intensities()[:, :, z_layer]


print('Calculating diffuse scattering for last snapshot ...')
I_single = calc_diffuse_hk0(
    history[-1], frac_coords, cell, box, grid,
    up_element=up_element, down_element=down_element
)
I_single = I_single / I_single.max()

extent = [grid.lower_limits[0], -grid.lower_limits[0],
          grid.lower_limits[1], -grid.lower_limits[1]]

plt.figure(figsize=(5, 5))
plt.imshow(I_single.T, origin='lower', cmap='Greys', clim=[0, 0.03], extent=extent)
plt.colorbar(label='Intensity (normalised)')
plt.xlabel('h  (r.l.u.)')
plt.ylabel('k  (r.l.u.)')
plt.title('Diffuse scattering hk0 — single snapshot')
plt.tight_layout()
plt.show()

### Averaged over the trajectory

A single snapshot is noisy because it represents one realisation of the disorder.  Averaging the intensity over many independent snapshots washes out the single-snapshot noise and reveals the underlying diffuse signal.

In [ ]:
print(f'Averaging over {len(history)} snapshots ...')

I_avg = np.zeros((grid.no_pixels[0], grid.no_pixels[1]))

for k, snap in enumerate(history):
    I_avg += calc_diffuse_hk0(
        snap, frac_coords, cell, box, grid,
        up_element=up_element, down_element=down_element
    )
    if (k + 1) % 5 == 0:
        print(f'  {k+1}/{len(history)}')

I_avg /= len(history)
I_avg /= I_avg.max()

print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, img, title in zip(
    axes,
    [I_single, I_avg],
    ['Single snapshot', f'Average  ({len(history)} snapshots)']
):
    im = ax.imshow(img.T, origin='lower', cmap='Greys', clim=[0, 0.3], extent=extent)
    ax.set_xlabel('h  (r.l.u.)')
    ax.set_ylabel('k  (r.l.u.)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Intensity')

plt.suptitle(f'Diffuse scattering hk0   T = {T_K:.0f} K  ({T:.4f} eV),  J₁ = {J_list[0]:.4f} eV', y=1.02)
plt.tight_layout()
plt.show()

## Section 4: Exploration

Use the cells below to run additional simulations with different parameters and compare the diffuse scattering.

**Suggested comparisons:**

1. **Sign of J₁**: flip the sign of `J_list[0]` and re-run.  Where do the diffuse peaks appear now?  Can you explain the shift in terms of the ordering wavevector?

2. **Temperature**: try `T = 0.2`, `T = 1.0`, `T = 5.0` (keeping J₁ fixed).  What happens to the sharpness and intensity of the diffuse features?

3. **Second-shell interaction**: set `J_list[1]` to a non-zero value (comparable to J₁).  What new features appear?

4. **Composition**: change `P_up` to 0.25 or 0.75 (75/25 mixture).  How does this affect the diffuse scattering?

5. **Resolution**: double `box[0]` and `box[1]` (keeping `box[2]` the same).  How does the quality of the averaged pattern improve?

In [ ]:
# ── Exploration cell — copy and modify as needed ─────────────────────────────
T_K_explore  = 2000.0                          # ← temperature in Kelvin
T_explore    = T_K_explore * kB                 # convert to eV
J_explore    = np.array([-J_list[0], 0.0])      # flipped sign as an example
P_up_explore = 0.5

spins_ex, _ = initialise(N_ats, P_up_explore, J_explore, connectivity_lists)
E_ex = calc_E(spins_ex, N_ats, P_up_explore, J_explore, connectivity_lists)

for _ in range(N_equil):
    spins_ex, dE = monte_carlo_run(spins_ex, connectivity_lists, J_explore, T_explore, N_moves)
    E_ex += dE

history_ex = []
for step in range(N_prod):
    spins_ex, dE = monte_carlo_run(spins_ex, connectivity_lists, J_explore, T_explore, N_moves)
    E_ex += dE
    if step % save_every == 0:
        history_ex.append(spins_ex.copy())

I_avg_ex = np.zeros((grid.no_pixels[0], grid.no_pixels[1]))
for snap in history_ex:
    I_avg_ex += calc_diffuse_hk0(
        snap, frac_coords, cell, box, grid,
        up_element=up_element, down_element=down_element
    )
I_avg_ex /= len(history_ex)
I_avg_ex /= I_avg_ex.max()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, img, ttl in zip(
    axes,
    [I_avg, I_avg_ex],
    [f'J₁ = {J_list[0]:.4f} eV,  T = {T_K:.0f} K',
     f'J₁ = {J_explore[0]:.4f} eV,  T = {T_K_explore:.0f} K']
):
    im = ax.imshow(img.T, origin='lower', cmap='Greys', clim=[0, 0.3], extent=extent)
    ax.set_xlabel('h  (r.l.u.)'); ax.set_ylabel('k  (r.l.u.)')
    ax.set_title(ttl)
    plt.colorbar(im, ax=ax, label='Intensity')
plt.tight_layout()
plt.show()

## Questions

1. Describe the diffuse scattering pattern for your material.  Where are the diffuse peaks?  What does their position in the hk0 plane tell you about the real-space ordering?

2. How many snapshots are needed before the averaged pattern stops changing significantly?  What determines this number?

3. Compare the single-snapshot and averaged patterns: which features are real (appear in both) and which are noise (only in the single snapshot)?

4. At what temperature does the diffuse scattering become negligible (i.e. the material is essentially disordered)?  How does this relate to the ordering transition temperature you found in Seminar 6?

5. If you had experimental diffuse scattering data for your material, how would you use the MC simulation to reproduce it?  What parameters would you adjust?